In [51]:
from dotenv import load_dotenv
load_dotenv()

from langchain.agents import create_agent
from langchain_google_genai import ChatGoogleGenerativeAI

def get_weather(city: str) -> str:
    """ Get a weather for a city."""
    return f"The weather in {city} is sunny."


In [52]:
# model = ChatGoogleGenerativeAI(model="gemini-3.6-flash") # 20 request/day

model = ChatGoogleGenerativeAI(model="gemini-3.7-flash", temperature=0)
agent = create_agent(model=model, tools=[get_weather], system_prompt="You are a helpful assistant.")

In [53]:
response = agent.invoke({"messages": [{"role": "user", "content": "What is the weather like in New York?"}]})
response["messages"]

[HumanMessage(content='What is the weather like in New York?', additional_kwargs={}, response_metadata={}, id='3a35f600-93ec-4063-b47d-2edaffb58806'),
 AIMessage(content=[], additional_kwargs={'function_call': {'name': 'get_weather', 'arguments': '{"city": "New York"}'}, '__gemini_function_call_thought_signatures__': {'call_9446424': 'Eq4CCqsCARFNMg/4LGYqUPASWuMCV5gq7et71Z8ruo+Qk9n+orwxpj/w7wkuyTLlfKQKnY39AyDz/Rwdz0xWzwj38Nqg8cxOdz2ujRskyYSTZf0hlmTAAq0kslZhQjcsRF3O9Eq/7xYAGBXOXklvrJ2TtgMOum7Y48sF/cobXHreWH/FSVlCY/dHz1Da5qW1o1hIUKRQkKSEzrlBklNAF1Rx/aul/mYp8aIDNbR5fmNDOMXBAVr8GKPYEhlLxl2TIdia/SOi99yLuYr5y6O0qqEeWyyTfJmvIiV0g4+6bacpc0zPbgirnq1XPc5SCUCGiixgZCDnvGOYX7p2t6tgueDmIIUtgGDQMVvfQxrVmwGNcgCvLaBn+OUs9CRJixG+TTW3dqM5WGM3TKV01rkWOvk='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.7-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a0a2bb-182c-7542-85ca-fec9fd362a54-0', tool_calls=[{'name': 'get_weather', 'args': {'city': 'New York

In [54]:
print(response["messages"][-1].content[0]['text'])

The weather in New York is currently sunny.


In [55]:
for chunk in agent.stream({'messages': [{'role': 'user', 'content': 'What is the weather in Boston'}]}):
    print(chunk)

{'model': {'messages': [AIMessage(content=[], additional_kwargs={'function_call': {'name': 'get_weather', 'arguments': '{"city": "Boston"}'}, '__gemini_function_call_thought_signatures__': {'call_8317587': 'EoUCCoICARFNMg8mcdutUU4veSctdkxnob8MIdh19Te+YlJ5CtUqqNJrxVhiieVdCsdN7wWCAeJA18YKU4Sr4AjzS2ncn2Tot7vYWREsx1M4nBQMALhOgRirZmDzNsHpLQr0zF3BaL9WqpluQgQoXzldbhiKe+U3vBdcsnJFYkE5IvW3XilpL4PIBKJAW7UXAduTbjg/HlampQOg6Vy/lIeDlm4yJAOUMPuDF6+XJZJVpukFRbDrFLebkC7g5PJ6neCRoXganDDKExOfiRJSUBzAQ9NIbBrGYTZk70nZ/olZqHz0byn3TFEqcLFFce5K6c0xNeTUsnb6u8gL/x3YJbGSHBavrS0/'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.7-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a0a2bb-4597-7a30-8bb2-8c3cf06b4b75-0', tool_calls=[{'name': 'get_weather', 'args': {'city': 'Boston'}, 'id': 'call_8317587', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 57, 'output_tokens': 59, 'total_tokens': 116, 'input_token_details': {'cache_read': 

In [ ]:
# It's taking doing to many request 
# for chunk in agent.stream({"messages": [{"role": "user", "content": "What is the weather in Boston"}]}):
#     # Each chunk may contain model/tool messages
#     for message in chunk.get("model", {}).get("messages", []):
#         for item in message.content:
#             if item.get("type") == "text":
#                 print(item["text"])
#     for tool_msg in chunk.get("tools", {}).get("messages", []):
#         print(tool_msg.content)


GoogleRateLimitError: Error calling model 'gemini-3.7-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 5, model: gemini-3.7-flash\nPlease retry in 20.183602713s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerMinutePerProjectPerModel-FreeTier',

In [ ]:

from IPython.core import events
async for event in agent.astream_events({"messages": [{"role": "user", "content": "What is the weather in Boston"}]}, version="v2"):
    if event["event"] == "on_chat_model_stream":
        print(event["data"]["chunk"].content, end="", flush=True)
    elif event["event"] == "on_tool_start":
        print(f"\n[calling tool: {event['name']}]")
    elif event["event"] == "on_tool_end":
        print(f"[tool result: {event['data']['output']}]")

GoogleRateLimitError: Error calling model 'gemini-3.6-flash' (Too Many Requests): 429 Too Many Requests. {'message': '{\n  "error": {\n    "code": 429,\n    "message": "You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.6-flash\\nPlease retry in 27.819395961s.",\n    "status": "RESOURCE_EXHAUSTED",\n    "details": [\n      {\n        "@type": "type.googleapis.com/google.rpc.Help",\n        "links": [\n          {\n            "description": "Learn more about Gemini API quotas",\n            "url": "https://ai.google.dev/gemini-api/docs/rate-limits"\n          }\n        ]\n      },\n      {\n        "@type": "type.googleapis.com/google.rpc.QuotaFailure",\n        "violatio